# Mel-Spectrogram CNN (multi-feature variant)

6-emotion CNN trained on **CREMA-D + RAVDESS** (TESS, SAVEE, and Surprised removed).

Each clip becomes a **5-channel 2D input** of shape `(n_mels, frames, 5)`:

| channel | feature | shape | role |
|---|---|---|---|
| 0 | log-mel-spectrogram | `(128, T)` | main timbre/energy image |
| 1 | zero-crossing rate | broadcast `(128, T)` | noisiness / frication |
| 2 | pitch (f0, YIN) | broadcast `(128, T)` | intonation |
| 3 | spectral centroid | broadcast `(128, T)` | brightness |
| 4 | RMS energy | broadcast `(128, T)` | loudness / arousal |

The four per-frame features are 1D (one value per time frame), so they're tiled across the mel axis to line up with the spectrogram.

In [ ]:
import re, os, sys, glob
from pathlib import Path
import numpy as np
import pandas as pd

ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB  = "google.colab" in sys.modules
if not ON_COLAB:
    try:
        import google.colab
        ON_COLAB = True
    except ImportError:
        pass

if ON_KAGGLE:
    OUT_DIR = Path("/kaggle/working")
elif ON_COLAB:
    OUT_DIR = Path("/content")
else:
    OUT_DIR = Path(".")

KNOWN_EMOTIONS = {"Angry", "Disgusted", "Fearful", "Happy", "Neutral", "Sad"}

def find_emotions_dir(root):
    for c in glob.glob(os.path.join(str(root), "**", ""), recursive=True):
        try:
            subs = {d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))}
        except OSError:
            continue
        if len(KNOWN_EMOTIONS & subs) >= 3:
            return Path(c)
    return None

CANDIDATES = [
    Path("/kaggle/input"),
    Path("/content/Emotions"),
    Path("Emotions"),
    Path("../Emotions"),
    Path.home() / "Downloads" / "Emotions",
]
EMO_DIR = None
for candidate in CANDIDATES:
    if candidate.exists():
        found = find_emotions_dir(candidate)
        if found:
            EMO_DIR = found
            break

if EMO_DIR is None and ON_COLAB:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("uldisvalainis/audio-emotions")
    EMO_DIR = find_emotions_dir(DATA_ROOT)

assert EMO_DIR is not None, (
    "Emotions folder not found.\n"
    "Kaggle: attach uldisvalainis/audio-emotions via Add Input.\n"
    "Colab : upload Emotions/ to /content/ or set up kaggle.json."
)
print(f"Environment : {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")
print(f"Dataset     : {EMO_DIR.resolve()}")

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rows.append({"filename": wav.name, "emotion": emo_path.name,
                     "source": source_of(wav.name), "path": str(wav)})
df = pd.DataFrame(rows)
df = df[df["source"].isin(["CREMA-D", "RAVDESS"])]
df = df[df["emotion"] != "Suprised"].reset_index(drop=True)
print(f"{len(df)} clips, {df['emotion'].nunique()} emotions")
df["emotion"].value_counts()

In [ ]:
import gc
import librosa
from tqdm.auto import tqdm

SR         = 16000
N_FFT      = 512
HOP        = 512
N_BINS     = N_FFT // 2 + 1
MAX_FRAMES = 110          # 110 × 512 / 16000 ≈ 3.5 s
FREQS      = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)

def _fit2d(m):
    if m.shape[1] < MAX_FRAMES:
        return np.pad(m, ((0, 0), (0, MAX_FRAMES - m.shape[1])))
    return m[:, :MAX_FRAMES]

def _fit1d(a):
    a = a[:MAX_FRAMES] if len(a) >= MAX_FRAMES else np.pad(a, (0, MAX_FRAMES - len(a)))
    return np.tile(a, (N_BINS, 1))

def features(path):
    y, sr = librosa.load(path, sr=SR)
    S    = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2
    perc = librosa.perceptual_weighting(S, FREQS)
    zcr  = librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0]
    cen  = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP)[0]
    rms  = librosa.feature.rms(y=y, hop_length=HOP)[0]
    return np.stack([_fit2d(perc), _fit1d(zcr), _fit1d(cen), _fit1d(rms)], axis=-1)

CACHE = OUT_DIR / "melcnn_cache.npz"
if CACHE.exists():
    dat    = np.load(str(CACHE))
    X      = dat["X"]
    labels = dat["labels"]
    print("loaded from cache:", X.shape)
else:
    paths  = list(df["path"])
    X      = np.empty((len(paths), N_BINS, MAX_FRAMES, 4), dtype="float32")
    for i, p in enumerate(tqdm(paths, desc="features")):
        X[i] = features(p)
        if i % 500 == 0:
            gc.collect()
    labels = df["emotion"].values
    np.savez_compressed(str(CACHE), X=X, labels=labels)
    print("extracted + cached:", X.shape)

print("X:", X.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()
y_int = le.fit_transform(labels)
y = to_categorical(y_int)
print("classes:", list(le.classes_))

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_int)

# Per-channel standardization using TRAIN stats only (each feature has a different scale)
axes = (0, 1, 2)                                   # over samples, mel, time -> one stat per channel
mean = Xtr.mean(axis=axes, keepdims=True)
std  = Xtr.std(axis=axes, keepdims=True)
Xtr = (Xtr - mean) / (std + 1e-9)
Xte = (Xte - mean) / (std + 1e-9)
print("train:", Xtr.shape, " test:", Xte.shape)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)

model = keras.Sequential([
    keras.Input(shape=Xtr.shape[1:]),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.BatchNormalization(), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.BatchNormalization(), layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.BatchNormalization(), layers.MaxPooling2D(2),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(y.shape[1], activation="softmax", dtype="float32"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    Xtr, ytr, validation_data=(Xte, yte),
    epochs=30, batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

pred = model.predict(Xte).argmax(1)
true = yte.argmax(1)
print(classification_report(true, pred, target_names=le.classes_))

cm = confusion_matrix(true, pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Mel+ZCR+pitch+centroid CNN")
plt.show()